# Notebook 03 — Advanced Modeling

**Models:** Random Forest → LightGBM (Optuna-tuned) → XGBoost → Keras 1D-CNN → Keras LSTM  
**Validation:** GroupKFold (n=5) by engine unit  
**Tracking:** MLflow  
**Analysis:** RUL clipping experiment, error analysis, SHAP, cost analysis, prediction intervals

In [ ]:
import sys; sys.path.insert(0, '..')

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow
import seaborn as sns

from src.data import load_processed, load_test_rul, get_last_cycle
from src.evaluate import (
    rmse, mae, nasa_score, evaluate_on_test,
    plot_predicted_vs_actual, plot_residuals, plot_rul_trajectories,
    cost_analysis, run_to_failure_cost, fixed_schedule_cost,
    plot_cost_vs_threshold,
)
from src.explain import explain_model
from src.features import load_feature_names
from src.train import (
    train_model, tune_lightgbm,
    build_sliding_windows, build_cnn_model, build_lstm_model, train_keras_model,
    train_quantile_models, predict_with_intervals,
    save_model, load_model,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('Imports OK')

---
## 1. Load processed features

In [ ]:
train_clip   = load_processed('train_FD001_clipped',   Path('../data/processed'))
train_unclip = load_processed('train_FD001_unclipped', Path('../data/processed'))
test_feat    = load_processed('test_FD001',             Path('../data/processed'))
rul_test     = load_test_rul('FD001')
feature_cols = load_feature_names('FD001', Path('../data/processed'))

X_clip   = train_clip[feature_cols].values
y_clip   = train_clip['rul'].values.astype(float)
X_unclip = train_unclip[feature_cols].values
y_unclip = train_unclip['rul'].values.astype(float)
groups   = train_clip['unit'].values

# Test set: last cycle per engine
test_last = get_last_cycle(test_feat)
X_test    = test_last[feature_cols].values
y_test    = rul_test.values.astype(float)

print(f'Train shape (clipped): {X_clip.shape}  y range: {y_clip.min():.0f}–{y_clip.max():.0f}')
print(f'Test  shape           : {X_test.shape}  y range: {y_test.min():.0f}–{y_test.max():.0f}')

---
## 2. Random Forest

In [ ]:
rf_params = {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 5}

print('Training Random Forest (clipped) …')
rf_result_clip = train_model(
    'rf', pd.DataFrame(X_clip, columns=feature_cols), y_clip, groups,
    params=rf_params,
    experiment_name='predictive-maintenance',
    run_name='rf_clipped',
    models_dir=MODELS_DIR,
)
cv = rf_result_clip['cv_metrics']
print(f'  CV RMSE  : {cv["rmse_mean"]:.1f} ± {cv["rmse_std"]:.1f}')
print(f'  CV MAE   : {cv["mae_mean"]:.1f}')
print(f'  NASA     : {cv["nasa_score_mean"]:.0f}')

In [ ]:
# Test-set evaluation
rf_test = evaluate_on_test(rf_result_clip['model'],
                           test_last, y_test, feature_cols)
print(f'RF test-set  RMSE={rf_test["rmse"]:.1f}  MAE={rf_test["mae"]:.1f}  NASA={rf_test["nasa_score"]:.0f}')

---
## 3. LightGBM — Optuna tuning

In [ ]:
print('Running Optuna tuning for LightGBM (40 trials) …')
tune_result = tune_lightgbm(
    pd.DataFrame(X_clip, columns=feature_cols), y_clip, groups,
    n_trials=40,
    experiment_name='predictive-maintenance',
)
best_params = tune_result['best_params']
print(f'Best RMSE: {tune_result["best_rmse"]:.2f}')
print(f'Best params: {best_params}')

In [ ]:
print('Training best LightGBM …')
lgbm_result_clip = train_model(
    'lgbm', pd.DataFrame(X_clip, columns=feature_cols), y_clip, groups,
    params=best_params,
    experiment_name='predictive-maintenance',
    run_name='lgbm_clipped',
    models_dir=MODELS_DIR,
)
cv = lgbm_result_clip['cv_metrics']
print(f'  CV RMSE  : {cv["rmse_mean"]:.1f} ± {cv["rmse_std"]:.1f}')
print(f'  NASA     : {cv["nasa_score_mean"]:.0f}')

lgbm_test = evaluate_on_test(lgbm_result_clip['model'],
                              test_last, y_test, feature_cols)
print(f'LightGBM test  RMSE={lgbm_test["rmse"]:.1f}  NASA={lgbm_test["nasa_score"]:.0f}')

In [ ]:
# Clipping experiment: unclipped vs clipped
print('LightGBM — unclipped target …')
lgbm_result_unclip = train_model(
    'lgbm', pd.DataFrame(X_unclip, columns=feature_cols), y_unclip, groups,
    params=best_params,
    experiment_name='predictive-maintenance',
    run_name='lgbm_unclipped',
    models_dir=MODELS_DIR,
)
lgbm_test_unclip = evaluate_on_test(lgbm_result_unclip['model'],
                                     test_last, y_test, feature_cols)
print(f'Unclipped  RMSE={lgbm_test_unclip["rmse"]:.1f}  NASA={lgbm_test_unclip["nasa_score"]:.0f}')
print(f'Clipped    RMSE={lgbm_test["rmse"]:.1f}  NASA={lgbm_test["nasa_score"]:.0f}')
print()
print('NASA score is typically much lower with clipping because the model stops')
print('over-predicting RUL during the uninformative early phase of engine life.')

---
## 4. XGBoost

In [ ]:
xgb_params = {'n_estimators': 500, 'learning_rate': 0.05,
              'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.8}
print('Training XGBoost …')
xgb_result = train_model(
    'xgb', pd.DataFrame(X_clip, columns=feature_cols), y_clip, groups,
    params=xgb_params,
    experiment_name='predictive-maintenance',
    run_name='xgb_clipped',
    models_dir=MODELS_DIR,
)
xgb_test = evaluate_on_test(xgb_result['model'], test_last, y_test, feature_cols)
print(f'XGBoost test  RMSE={xgb_test["rmse"]:.1f}  NASA={xgb_test["nasa_score"]:.0f}')

---
## 5. Deep learning — 1D-CNN and LSTM

In [ ]:
print('Building sliding windows for sequence models (window=30) …')
X_seq, y_seq, groups_seq = build_sliding_windows(
    train_clip, feature_cols, window=30, target_col='rul'
)
print(f'Sequence shape: {X_seq.shape}  (N, window, n_features)')

In [ ]:
input_shape = (X_seq.shape[1], X_seq.shape[2])

print('Training 1D-CNN …')
cnn_model = build_cnn_model(input_shape)
cnn_result = train_keras_model(
    cnn_model, X_seq, y_seq, groups_seq,
    experiment_name='predictive-maintenance',
    run_name='cnn',
    epochs=60, batch_size=256, patience=10,
    models_dir=MODELS_DIR,
)
cv = cnn_result['cv_metrics']
print(f'  CNN CV RMSE: {cv["rmse_mean"]:.1f} ± {cv["rmse_std"]:.1f}')

In [ ]:
print('Training LSTM …')
lstm_model = build_lstm_model(input_shape)
lstm_result = train_keras_model(
    lstm_model, X_seq, y_seq, groups_seq,
    experiment_name='predictive-maintenance',
    run_name='lstm',
    epochs=60, batch_size=256, patience=10,
    models_dir=MODELS_DIR,
)
cv = lstm_result['cv_metrics']
print(f'  LSTM CV RMSE: {cv["rmse_mean"]:.1f} ± {cv["rmse_std"]:.1f}')

---
## 6. Error analysis and visualisations

In [ ]:
best_model = lgbm_result_clip['model']  # use LightGBM as primary
y_pred_test = lgbm_test['y_pred']

# Predicted vs actual
fig1 = plot_predicted_vs_actual(
    y_test, y_pred_test,
    title='LightGBM — Predicted vs Actual RUL (test set)',
    save_path=FIGURES_DIR / 'fig_pred_vs_actual.png',
)
plt.show()

# Residuals
fig2 = plot_residuals(
    y_test, y_pred_test,
    title='LightGBM — Residual distribution (test set)',
    save_path=FIGURES_DIR / 'fig_residuals.png',
)
plt.show()

# RUL trajectories for 4 engines
fig3 = plot_rul_trajectories(
    test_feat, best_model, feature_cols,
    unit_ids=[1, 5, 10, 20],
    y_true_rul=rul_test,
    title='LightGBM — RUL trajectories (test engines)',
    save_path=FIGURES_DIR / 'fig_rul_trajectories.png',
)
plt.show()
print('Evaluation figures saved.')

---
## 7. SHAP explainability

In [ ]:
# Use validation fold for SHAP (representative sample, not test set)
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)
tr_idx, val_idx = next(iter(gkf.split(X_clip, y_clip, groups=groups)))
X_val_shap = pd.DataFrame(X_clip[val_idx], columns=feature_cols)
y_val_shap = y_clip[val_idx]
y_pred_val = np.clip(best_model.predict(X_clip[val_idx]), 0, None)

print('Computing SHAP values …')
shap_result = explain_model(
    best_model, X_val_shap, feature_cols,
    y_true=y_val_shap, y_pred=y_pred_val,
    save_dir=FIGURES_DIR,
    dataset='FD001',
)
print('Top features:', shap_result['top_features'][:5])
print()
print(shap_result['interpretation'])

---
## 8. Business cost analysis

In [ ]:
thresholds = np.arange(0, 151, 1, dtype=float)
cost_df = cost_analysis(
    y_test, y_pred_test,
    thresholds=thresholds,
    cost_failure=10_000.0,
    cost_maintenance=1_000.0,
    cost_per_early_cycle=50.0,
)

optimal_idx = cost_df['total_cost'].idxmin()
optimal_t   = cost_df.loc[optimal_idx, 'threshold']
optimal_cost = cost_df.loc[optimal_idx, 'total_cost']

baseline_rtf   = run_to_failure_cost(len(y_test), cost_failure=10_000.0)
baseline_fixed = fixed_schedule_cost(y_test, fixed_threshold=50)

saving_rtf   = 100 * (baseline_rtf   - optimal_cost) / baseline_rtf
saving_fixed = 100 * (baseline_fixed - optimal_cost) / baseline_fixed

print(f'Optimal threshold  : T = {optimal_t:.0f} cycles')
print(f'Model cost         : ${optimal_cost:,.0f}')
print(f'Run-to-failure     : ${baseline_rtf:,.0f}  (saving: {saving_rtf:.1f}%)')
print(f'Fixed schedule (50): ${baseline_fixed:,.0f}  (saving: {saving_fixed:.1f}%)')

In [ ]:
fig_cost = plot_cost_vs_threshold(
    cost_df, optimal_t, baseline_rtf, baseline_fixed,
    save_path=FIGURES_DIR / 'cost_vs_threshold.png',
)
plt.show()
print('Cost analysis figure saved.')

---
## 9. Prediction intervals (quantile regression + conformal)

In [ ]:
print('Training quantile regressors …')
quantile_result = train_quantile_models(
    pd.DataFrame(X_clip, columns=feature_cols), y_clip, groups,
    alpha_low=0.1, alpha_high=0.9,
    experiment_name='predictive-maintenance',
    models_dir=MODELS_DIR,
)
print(f'Calibration coverage: {quantile_result["coverage"]:.1%}  (target: ~80%)')
print(f'Conformal margin    : {quantile_result["conf_margin"]:.1f} cycles')

In [ ]:
y_hat, lower, upper = predict_with_intervals(
    quantile_result['point_model'],
    quantile_result['lower_model'],
    quantile_result['upper_model'],
    X_test,
    conf_margin=quantile_result['conf_margin'],
)

empirical_coverage = float(np.mean((y_test >= lower) & (y_test <= upper)))
print(f'Test-set coverage: {empirical_coverage:.1%}')
assert 0.65 <= empirical_coverage <= 0.95, f'Coverage {empirical_coverage:.1%} out of expected range'

# Plot trajectories with uncertainty bands
fig_unc = plot_rul_trajectories(
    test_feat, quantile_result['point_model'], feature_cols,
    unit_ids=[1, 5, 15],
    y_true_rul=rul_test,
    title='RUL with 80% prediction intervals',
    save_path=FIGURES_DIR / 'fig_rul_uncertainty.png',
)
plt.show()

---
## 10. Final model comparison table

In [ ]:
results_rows = [
    {'Model': 'Random Forest (clipped)',    'Test RMSE': rf_test['rmse'],               'Test MAE': rf_test['mae'],               'Test NASA': rf_test['nasa_score']},
    {'Model': 'LightGBM (clipped)',         'Test RMSE': lgbm_test['rmse'],             'Test MAE': lgbm_test['mae'],             'Test NASA': lgbm_test['nasa_score']},
    {'Model': 'LightGBM (unclipped)',       'Test RMSE': lgbm_test_unclip['rmse'],      'Test MAE': lgbm_test_unclip['mae'],      'Test NASA': lgbm_test_unclip['nasa_score']},
    {'Model': 'XGBoost (clipped)',          'Test RMSE': xgb_test['rmse'],              'Test MAE': xgb_test['mae'],              'Test NASA': xgb_test['nasa_score']},
]

results_df = pd.DataFrame(results_rows)
results_df['Test RMSE'] = results_df['Test RMSE'].round(1)
results_df['Test MAE']  = results_df['Test MAE'].round(1)
results_df['Test NASA'] = results_df['Test NASA'].round(0).astype(int)
results_df = results_df.sort_values('Test RMSE')
print(results_df.to_string(index=False))

# Save for README
results_df.to_csv('../reports/model_results_FD001.csv', index=False)
print('\nResults saved to reports/model_results_FD001.csv')

---
## When does deep learning beat tree models on this task?

The 1D-CNN and LSTM process the raw sensor sequence directly. In principle,
sequence models can capture ordering effects that hand-crafted rolling features
might miss. In practice on FD001 (one operating condition, one fault mode),
the advantage is marginal because:

1. The rolling-window and slope features already encode most temporal structure.
2. FD001's degradation is fairly smooth — no sharp transitions that would
   require long-range memory.
3. Tree models have a regularisation advantage on tabular data (split-based
   feature selection, no gradient vanishing).

On FD002/FD004 with multiple operating conditions and more variability, LSTMs
tend to improve more relative to tree models. In any case, a well-tuned
gradient-boosted model should be your first production candidate; deep learning
is an enhancement, not a replacement.